In [1]:
import wandb
import sys
import matplotlib.pyplot as plt
import scprep
import pandas as pd
sys.path.append('../hypgs/models')
from omegaconf import OmegaConf
from train_ae import load_data, make_model
import numpy as np
import os
import glob
import scanpy as sc
os.chdir('/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/hypergraph_scattering/hypgs/models')
import torch
from plotly3d.plot import scatter
# os.chdir('/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/hypergraph_scattering/hypgs/models')
import phate

In [2]:
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist, squareform
import graphtools
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import torch
import torch_geometric
import scipy.sparse as sp
from torch_geometric.data import Data
from tqdm import tqdm

/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/torch_geometric/typing.py:110: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


In [3]:
# Initialize wandb (replace 'your_entity' and 'your_project' with your specific details)
wandb.login()
api = wandb.Api()

# Specify your entity, project, and sweep ID
entity = "xingzhis"
project = "dmae"
sweep_id = 'oxg8o0nu'

# Fetch the sweep
sweep = api.sweep(f"{entity}/{project}/{sweep_id}")
run_ids = [run.id for run in sweep.runs]

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: xingzhis. Use `wandb login --relogin` to force relogin


In [4]:
section_df = pd.read_csv('/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/sea_ad/data/hypergraphae/braak.csv')
labels, unique = pd.factorize(section_df['Braak'], sort=True)
section_df['braak_code'] = labels
section_dict = {section: braak for section, braak in zip(section_df['Section'], section_df['braak_code'])}

In [11]:
datas = []
# run = [run for run in sweep.runs if run.id == run_ids[0]][0]
for run in tqdm(sweep.runs):
    cfg = OmegaConf.create(run.config)
    section = cfg.data.name
    try:
        braak = section_dict[section.strip('.h5ad')]
    except:
        print(f'Warning: {section.strip(".h5ad")} Braak not found.')
        continue
    folder_path = "wandb/"
    folder_list = glob.glob(f"{folder_path}*{run.id}*")
    try:
        ckpt_files = glob.glob(f"{folder_list[0]}/files/*.ckpt")
        ckpt_path = ckpt_files[0]
    except:
        print(f'Warning: {section.strip(".h5ad")} checkpoint not found.')
        continue
    trainloader, valloader, X, phate_coords, colors, dist = load_data(cfg)
    emb_dim = cfg.model.emb_dim
    data_path = os.path.join(cfg.data.root, cfg.data.name + cfg.data.filetype)
    data = np.load(data_path, allow_pickle=True)
    dist_std = np.std(data['dist'].flatten())
    model = make_model(cfg, X.shape[1], emb_dim, dist_std, from_checkpoint=True, checkpoint_path=ckpt_path)
    model.eval()
    x_all = torch.tensor(data['data'][data['is_train']]).float()
    x_pred, z_pred = model(x_all)
    x_pred = x_pred.detach().cpu().numpy()
    z_pred = z_pred.detach().cpu().numpy()
    section = cfg.data.name
    # classes = np.load(f'../../data/hgsae/{section}_label.npy', allow_pickle=True)
    classes = data['colors']
    classes = classes[data['is_train']]

    X = z_pred
    kmeans = KMeans(random_state=0)
    kmeans.fit(X)
    cluster_labels = kmeans.labels_

    label_cluster_df = pd.DataFrame({'label': classes, 'cluster': cluster_labels})
    label_cluster_table = pd.crosstab(label_cluster_df['label'], label_cluster_df['cluster']).T
    label_cluster_table = label_cluster_table.reindex(columns=all_labels, fill_value=0)
    label_cluster_table_norm = label_cluster_table.div(label_cluster_table.sum(axis=1), axis=0)

    centroids = kmeans.cluster_centers_
    G = graphtools.Graph(centroids, knn=3, random_state=0)
    G_K = G.K

    if not sp.isspmatrix_coo(G_K):
        G_K = sp.coo_matrix(G_K)

    # Extract edges and weights from the sparse matrix
    row = torch.tensor(G_K.row, dtype=torch.long)
    col = torch.tensor(G_K.col, dtype=torch.long)
    edge_index = torch.stack([row, col], dim=0)

    # Edge weights
    edge_weights = torch.tensor(G_K.data, dtype=torch.float)
    x = torch.tensor(label_cluster_table_norm.values, dtype=torch.float)
    # Create a PyTorch Geometric data object with edge attributes
    print(section, braak)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_weights.unsqueeze(1), y=torch.tensor([braak]))
    datas.append(data)

  0%|          | 0/59 [00:00<?, ?it/s]

/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
  2%|▏         | 1/59 [00:01<01:30,  1.56s/it]

H20.33.004.Cx26.MTG.02.007.1.02.02.h5ad 4


/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
  3%|▎         | 2/59 [00:02<01:15,  1.33s/it]

H20.33.012.Cx24.MTG.02.007.1.03.02.h5ad 2


...

H20.33.035.Cx26.MTG.02.007.1.01.03.h5ad 3


/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
 98%|█████████▊| 58/59 [02:18<00:02,  2.12s/it]

H20.33.025.Cx28.MTG.02.007.1.01.02.h5ad 4


/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
100%|██████████| 59/59 [02:21<00:00,  2.39s/it]

H21.33.032.CX24.MTG.02.007.1.01.04.h5ad 3


In [12]:
torch.save(datas, '../../data/hgsae2/embedding_graph_dataset.pt')

In [7]:
len(datas)

53

In [8]:
datas

[Data(x=[8, 29], edge_index=[2, 38], edge_attr=[38, 1], y=[1]),
 Data(x=[8, 24], edge_index=[2, 38], edge_attr=[38, 1], y=[1]),
 Data(x=[8, 28], edge_index=[2, 38], edge_attr=[38, 1], y=[1]),
 Data(x=[8, 29], edge_index=[2, 36], edge_attr=[36, 1], y=[1]),
 Data(x=[8, 29], edge_index=[2, 36], edge_attr=[36, 1], y=[1]),
 Data(x=[8, 28], edge_index=[2, 32], edge_attr=[32, 1], y=[1]),
 Data(x=[8, 29], edge_index=[2, 40], edge_attr=[40, 1], y=[1]),
 Data(x=[8, 28], edge_index=[2, 38], edge_attr=[38, 1], y=[1]),
 Data(x=[8, 29], edge_index=[2, 32], edge_attr=[32, 1], y=[1]),
 Data(x=[8, 27], edge_index=[2, 40], edge_attr=[40, 1], y=[1]),
 Data(x=[8, 31], edge_index=[2, 32], edge_attr=[32, 1], y=[1]),
 Data(x=[8, 28], edge_index=[2, 40], edge_attr=[40, 1], y=[1]),
 Data(x=[8, 27], edge_index=[2, 36], edge_attr=[36, 1], y=[1]),
 Data(x=[8, 28], edge_index=[2, 38], edge_attr=[38, 1], y=[1]),
 Data(x=[8, 27], edge_index=[2, 40], edge_attr=[40, 1], y=[1]),
 Data(x=[8, 30], edge_index=[2, 36], edg